# ASE VASP smoke test

One-time smoke test to verify ASE+VASP configuration. It:
- copies example calculation inputs from `TMP/sc`
- rattles atomic positions
- writes VASP inputs to a temp workdir under `TMP`
- runs `run-calc.sh` via `ase.calculators.vasp.Vasp` using `get_potential_energy`

Requires a working `run-calc.sh` and `run-calc.conf` (in the workdir or `~/.hecss`).


In [1]:
#| vasp_ase
from pathlib import Path
from tempfile import TemporaryDirectory
import shutil
import os
import ase.io
from ase.calculators.vasp import Vasp

# Paths
source_dir = Path('../TMP/sc').resolve()
run_calc = Path('../run-calc.sh').resolve()

if not source_dir.exists():
    raise FileNotFoundError(f"Missing source directory: {source_dir}")
if not run_calc.exists():
    raise FileNotFoundError(f"Missing run-calc.sh: {run_calc}")

# Working directory in TMP
work_tmp = TemporaryDirectory(dir='../TMP')
workdir = Path(work_tmp.name)

# Copy required inputs; include local run-calc.conf if present for convenience
for fname in ['POSCAR', 'INCAR', 'POTCAR', 'KPOINTS']:
    shutil.copy(source_dir / fname, workdir / fname)
if (source_dir / 'run-calc.conf').exists():
    shutil.copy(source_dir / 'run-calc.conf', workdir / 'run-calc.conf')

# Load, rattle, and rewrite POSCAR
atoms = ase.io.read(workdir / 'POSCAR')
atoms.rattle(stdev=0.01, seed=1)
ase.io.write(workdir / 'POSCAR', atoms, format='vasp')

# Run VASP via ASE; use absolute path to run-calc.sh
cwd = Path.cwd()
os.chdir(workdir)
try:
    calc = Vasp(directory=str(workdir), command=f'{run_calc} "ase_smoke"', restart=False)
    atoms.calc = calc
    energy = atoms.get_potential_energy()
    print(f"Energy: {energy} eV")
    assert (workdir / 'OUTCAR').exists(), 'OUTCAR not produced'
    print(f"Workdir: {workdir}")
finally:
    os.chdir(cwd)



Energy: -55.65267549 eV
Workdir: ../TMP/tmpano7eqne
